<a href="https://colab.research.google.com/github/BarrenWuffet402/Ragify/blob/main/Ragify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM4BIZ CHATBOT (RAG FOR NOTES AND SLIDES FROM DAY 1 & 2, April 11th and 12th, 2026, Stanford)

This notebook builds a privacy-first RAG assistant for personal documents (PDF/TXT/MD/HTML/PPTX).

In [16]:
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate beautifulsoup4 python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.2 MB/s eta 0:00:00


In [17]:
import re
from pathlib import Path

import torch
import numpy as np
from pypdf import PdfReader
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files

print("GPU available:", torch.cuda.is_available())
from pptx import Presentation


GPU available: False


In [18]:
NOTE_DIR = Path("/content/Ragify/data/notes")

if NOTE_DIR.exists():
    note_files = sorted([p for p in NOTE_DIR.rglob("*") if p.suffix.lower() in [".html", ".htm", ".txt", ".md", ".pdf", ".pptx"]])
    print(f"Found {len(note_files)} note files in {NOTE_DIR}")
    for p in note_files[:10]:
        print("-", p.name)
    if len(note_files) > 10:
        print("...")
else:
    print(f"{NOTE_DIR} not found. Falling back to manual upload.")
    uploaded = files.upload()
    note_files = [Path(name) for name in uploaded.keys()]
    print("Uploaded:", [p.name for p in note_files])

Found 4 note files in /content/Ragify/data/notes
- TECH16_Day1_ClassNotes.html
- TECH16_Day2_ClassNotes.html
- slides.pdf
- slides.pptx


In [19]:
def read_file_text(path: Path) -> str:
    ext = path.suffix.lower()
    try:
        if ext == ".pdf":
            reader = PdfReader(str(path), strict=False)
            return chr(10).join((p.extract_text() or "") for p in reader.pages)
        if ext in [".txt", ".md"]:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
        if ext in [".html", ".htm"]:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                soup = BeautifulSoup(f.read(), "html.parser")
            return soup.get_text(" ", strip=True)
        if ext == ".pptx":
            prs = Presentation(str(path))
            texts = []
            for slide in prs.slides:
                for shape in slide.shapes:
                    if hasattr(shape, "text") and shape.text:
                        texts.append(shape.text)
            return "\n".join(texts)

        print(f"Skipping unsupported file type: {path.name}")
        return ""
    except Exception as e:
        print(f"Failed to read {path.name}: {type(e).__name__}: {e}")
        return ""

def chunk_text(text: str, chunk_size: int = 900, overlap: int = 150):
    text = re.sub(r"\\s+", " ", text).strip()
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

documents = []
for path in note_files:
    txt = read_file_text(path)
    if txt.strip():
        documents.append({"source": path.name, "text": txt})
    else:
        print(f"No usable text extracted: {path.name}")

chunks = []
for doc in documents:
    for c in chunk_text(doc["text"]):
        chunks.append({"source": doc["source"], "text": c})

print("Loaded docs:", [d["source"] for d in documents])
print("Total chunks:", len(chunks))

Loaded docs: ['TECH16_Day1_ClassNotes.html', 'TECH16_Day2_ClassNotes.html', 'slides.pdf', 'slides.pptx']
Total chunks: 357


In [20]:
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

chunk_texts = [c['text'] for c in chunks]
if not chunk_texts:
    raise ValueError('No text extracted. Upload at least one readable PDF/TXT/MD file.')

embeddings = embed_model.encode(chunk_texts, convert_to_numpy=True, normalize_embeddings=True)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity when embeddings are normalized
index.add(np.asarray(embeddings, dtype=np.float32))

print('FAISS index size:', index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index size: 357


In [21]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Faster default for Colab CPU

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto"
)

print("Model loaded:", MODEL_ID)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [22]:
def retrieve(query: str, k: int = 3):
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, idxs = index.search(np.asarray(q_emb, dtype=np.float32), k)
    results = []
    for score, i in zip(scores[0], idxs[0]):
        if i == -1:
            continue
        item = chunks[i]
        results.append({
            'rank': len(results) + 1,
            'score': float(score),
            'source': item['source'],
            'text': item['text']
        })
    return results

def answer_with_rag(question: str, k: int = 3, max_new_tokens: int = 120):
    hits = retrieve(question, k=k)
    context = '\n\n'.join([f"[{h['rank']}] Source: {h['source']}\n{h['text']}" for h in hits])

    prompt = f"""You are a helpful assistant for personal documents.
Answer only using the provided context.
If the answer is not in context, say: "I don't see that in the provided documents."

Question: {question}

Context:
{context}

Return:
1) A concise answer
2) Sources used like [1], [2]
"""

    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    gen_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return response, hits

def ask_question(question: str, k: int = 3):
    answer, hits = answer_with_rag(question, k=k)
    print('Q:', question)
    print('\nA:', answer)
    print('\nRetrieved chunks:')
    for h in hits:
        print(f"[{h['rank']}] {h['source']} (score={h['score']:.3f})")
    return answer, hits

In [23]:
demo_questions = [
    "Summarize the top 5 practical LLM takeaways from these notes with citations."
]

for q in demo_questions:
    print("=" * 80)
    ask_question(q, k=3)


KeyboardInterrupt: 

## Assignment Write-Up Notes
- Goal: Ask natural-language questions over personal files (PDF/TXT/MD/HTML/PPTX).
- Privacy approach: No external API calls to proprietary LLM endpoints.
- Method: Chunk documents -> embed with MiniLM -> retrieve with FAISS -> grounded answer generation.
- Limitation: Colab runtime is cloud-hosted; stricter privacy requires local runtime (e.g., laptop + Ollama).